# BoN vs PSP — generation cache + Best-of-N (Colab)

**Before you start:** Runtime → Change runtime type → **GPU** (A100 or L4 if you have Colab Pro, T4 otherwise).
Set `REPO_URL` in cell 4 to your GitHub repo. For a private repo, use a token URL or upload a zip of the repo to `MyDrive/ai6132_its/repo.zip`.

All results are written to Google Drive. If Colab disconnects, reconnect and rerun cells 1–4 and then the cell you were on. Finished prompts are skipped.

## 1. Mount Drive and set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJ  = '/content/drive/MyDrive/ai6132_its'          # project folder on Drive
CACHE = f'{PROJ}/cache/sd15_geneval_T64'             # keep 'geneval' in the name (PSP scripts glob for it)
RES   = f'{PROJ}/results_bon'
os.makedirs(PROJ, exist_ok=True)
print(os.listdir(PROJ))

## 2. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Install packages (~2–3 min)
These versions were tested together. ImageReward breaks with newer `transformers`, and it also needs OpenAI's `clip` package. Colab's preinstalled `peft` is removed because it breaks diffusers with this `transformers` version.
Ignore pip's dependency-resolver warnings, but **if the import check fails, use Runtime → Restart session, then rerun cells 1, 3 and 4**.

In [ ]:
!pip -q install "diffusers==0.31.0" "transformers==4.38.2" "huggingface_hub==0.25.2" "image-reward==1.5" "git+https://github.com/openai/CLIP.git"
!pip -q uninstall -y peft   # Colab's peft needs a newer transformers and breaks diffusers imports; we don't use it
import ImageReward, diffusers, transformers, torch
print('ok | torch', torch.__version__, '| diffusers', diffusers.__version__, '| transformers', transformers.__version__, '| cuda', torch.cuda.is_available())

## 4. Get the code (clone from GitHub, or unzip from Drive)

In [ ]:
REPO_URL = 'https://github.com/<your-username>/inference-scaling-seed-search.git'   # <- edit
CODE = '/content/inference-scaling-seed-search'
import os
!rm -rf {CODE}
if os.path.exists(f'{PROJ}/repo.zip'):
    !unzip -q -o {PROJ}/repo.zip -d /content/
else:
    !git clone -q {REPO_URL} {CODE}
%cd {CODE}
!git log --oneline -1 2>/dev/null; ls

## 5. Timing test (~5–15 min, mostly model downloads on the first run)
This runs 2 prompts × 32 seeds to local disk. The second prompt's time excludes warm-up.
If you hit CUDA out-of-memory on a T4, set `BATCH_SIZE=8`.

In [ ]:
!BATCH_SIZE=16 bash harness/run_timing_test.sh

**Decide how many prompts to run.** Keep 32 seeds, which are needed for N=32. Reduce the prompt count if the run won't fit in the time you have.
Set `NUM_PROMPTS` below. Use 120 if the estimate fits, otherwise 90 or 60. Cell 6 keeps an equal number of prompts per category automatically.

In [ ]:
NUM_PROMPTS = 120     # 120 = all; e.g. 60 -> 10 per category
BATCH_SIZE  = 16      # 8 if T4 runs out of memory

## 6. Full cache run (hours; resumable)
This writes to Drive, so a disconnect loses at most the current prompt. If the session drops, rerun cells 1, 3 and 4, then this one.
With fewer than 120 prompts, an equal number per category is kept so all 6 GenEval categories stay balanced.

In [ ]:
import json
src = 'data/geneval_subset_120.jsonl'
rows = [json.loads(l) for l in open(src)]
if NUM_PROMPTS < len(rows):
    per_tag = NUM_PROMPTS // 6
    by_tag = {}
    for r in rows: by_tag.setdefault(r['tag'], []).append(r)
    rows = [r for t in sorted(by_tag) for r in by_tag[t][:per_tag]]
    for i, r in enumerate(rows): r['prompt_id'] = i
src_run = f'{PROJ}/prompts_run.jsonl'
with open(src_run, 'w') as f:
    for r in rows: f.write(json.dumps(r) + '\n')
print(len(rows), 'prompts ->', src_run)

In [ ]:
!python harness/generate_cache.py \
  --prompts-path {src_run} \
  --output-dir {CACHE} \
  --num-seeds 32 --time-steps 64 --guidance-scale 7.5 \
  --batch-size {BATCH_SIZE} --decode-batch 4

**Expected per-prompt log line:**
```
[prompt 7] 32 seeds in 63.2s | traj-steps 2048 (= 32x64) | final IR mean -0.12 max 1.41 | 8/120 | ETA 118 min
```
- `traj-steps` is always 2048 (= 32 × 64). The run stops with an error if the counter disagrees.
- Per-prompt final IR means are noisy, with typical values between −1 and +1.

## 7. Progress check (run any time, even while 6 is running in another tab)

In [ ]:
import pandas as pd
log = pd.read_csv(f'{CACHE}/compute_log.csv')
print(f"{log.prompt_id.nunique()} prompts done | {log.sec_total.mean():.0f}s/prompt | all counts exact: {(log.traj_steps == log.expected_traj_steps).all()}")

## 8. Best-of-N seed-count sweep (no GPU needed, ~10 s)
Selections use non-overlapping seed groups: 32 groups at N=1, down to 1 group at N=32. The bootstrap run gives confidence intervals at large N.

In [ ]:
!python methods/bon/bon_replay.py --cache {CACHE} --prompts-path {src_run} --n-values 1,2,4,8,16,32 --out-dir {RES}
!python methods/bon/bon_replay.py --cache {CACHE} --prompts-path {src_run} --n-values 1,2,4,8,16,32 --bootstrap 200 --out-dir {RES}_bootstrap
from IPython.display import Image, display
display(Image(f'{RES}/bon_reward_vs_N.png'))
pd.read_csv(f'{RES}/bon_sweep.csv')

**Sanity anchors** come from the PSP paper (SD v1.5, T=64, all 553 GenEval prompts): standard sampling (N=1) has IR ≈ −0.16 and BoN N=4 has IR ≈ 0.66.
Your 120-prompt subset should land within a few tenths of these. The curve should rise steeply up to N≈8 and then flatten.
If N=1 is far from −0.16, check the model ID, guidance scale and steps before going further.

## 9. (Optional) Compute-budget sweep: seeds vs steps at equal compute
This needs extra caches at fewer steps. At equal compute C = N × T, it compares, for example, BoN-4 at T=64 with BoN-8 at T=32 and BoN-16 at T=16.
A T=32 cache costs about half of the T=64 run, and T=16 about a quarter. Run this on fewer prompts if time is short.

In [ ]:
for T in (32, 16):
    !python harness/generate_cache.py --prompts-path {src_run} --output-dir {PROJ}/cache/sd15_geneval_T{T} \
        --num-seeds 32 --time-steps {T} --batch-size {BATCH_SIZE} --decode-batch 4

In [ ]:
!python methods/bon/bon_replay.py --cache {CACHE} {PROJ}/cache/sd15_geneval_T32 {PROJ}/cache/sd15_geneval_T16 --out-dir {RES}_budget
display(Image(f'{RES}_budget/bon_reward_vs_C.png'))
pd.read_csv(f'{RES}_budget/bon_sweep.csv')

## 10. Handover files
This creates a small zip with everything except the images. The images stay on Drive; share the `cache/` folder with C and D.

In [ ]:
import shutil, os
H = f'{PROJ}/handover'; os.makedirs(H, exist_ok=True)
for f in ('metrics.csv', 'compute_log.csv', 'args.json'):
    shutil.copy(f'{CACHE}/{f}', f'{H}/T64_{f}')
shutil.copy(src_run, f'{H}/prompts_run.jsonl')
for d in (RES, f'{RES}_bootstrap', f'{RES}_budget'):
    if os.path.isdir(d): shutil.copytree(d, f'{H}/{os.path.basename(d)}', dirs_exist_ok=True)
shutil.copy('docs/harness.md', f'{H}/harness.md'); shutil.copy('eval/geneval_bridge.py', f'{H}/geneval_bridge.py')
shutil.make_archive(f'{PROJ}/handover_bon', 'zip', H)
print('Created', f'{PROJ}/handover_bon.zip'); print(sorted(os.listdir(H)))

## 11. Once C has GenEval results: add the held-out metric to the BoN table
C runs GenEval on the exported folder (`eval/geneval_bridge.py export`), then parses its results file to `geneval_per_image.csv`.

In [ ]:
GENEVAL_CSV = f'{PROJ}/geneval_per_image.csv'   # from C
!python methods/bon/bon_replay.py --cache {CACHE} --prompts-path {src_run} --geneval-csv {GENEVAL_CSV} --out-dir {RES}_geneval
pd.read_csv(f'{RES}_geneval/bon_sweep.csv')[['N','C_per_prompt','ir_mean','ir_std','geneval_mean','geneval_std']]